In [1]:
spark.table("silver_orders").select("OrderDate").printSchema()

StatementMeta(, 57f7e8f0-4526-437a-b6ea-65e138918c73, 3, Finished, Available, Finished, False)

root
 |-- OrderDate: timestamp (nullable = true)



In [3]:
from pyspark.sql import functions as F
from pyspark.sql.types import BinaryType
det = (spark.table("silver_order_details")
       .withColumn("UnitPrice", F.col("UnitPrice").cast("double"))
       .withColumn("Quantity",  F.col("Quantity").cast("int"))
       .withColumn("Discount",  F.col("Discount").cast("double")))

StatementMeta(, 57f7e8f0-4526-437a-b6ea-65e138918c73, 5, Finished, Available, Finished, False)

In [6]:
from pyspark.sql import functions as F

cust = spark.table("silver_customers")
prod = spark.table("silver_products")
cat  = spark.table("silver_categories")
sup  = spark.table("silver_suppliers")
emp  = spark.table("silver_employees")
ords = spark.table("silver_orders")
det  = spark.table("silver_order_details")

# Dimensions
dim_customer = cust.select(
    F.col("CustomerID").alias("customer_id"),
    F.col("CompanyName").alias("customer_name"),
    F.col("City").alias("customer_city"),
    F.col("Country").alias("customer_country"))

dim_product = (prod
    .join(cat.select("CategoryID", "CategoryName"), "CategoryID", "left")
    .join(sup.select("SupplierID", F.col("CompanyName").alias("supplier_name")), "SupplierID", "left")
    .select(F.col("ProductID").alias("product_id"),
            F.col("ProductName").alias("product_name"),
            F.col("CategoryName").alias("category"),
            "supplier_name"))

dim_employee = emp.select(
    F.col("EmployeeID").alias("employee_id"),
    F.concat_ws(" ", "FirstName", "LastName").alias("employee_name"),
    F.col("Title").alias("job_title"))

# Date dimension from the order date range
orders = ords.withColumn("order_date", F.to_date("OrderDate"))
r = orders.agg(F.min("order_date").alias("mn"), F.max("order_date").alias("mx")).first()
dim_date = (spark.sql(f"SELECT explode(sequence(to_date('{r.mn}'), to_date('{r.mx}'), interval 1 day)) AS date_key")
    .withColumn("year", F.year("date_key"))
    .withColumn("quarter", F.quarter("date_key"))
    .withColumn("month", F.month("date_key"))
    .withColumn("month_name", F.date_format("date_key", "MMM"))
    .withColumn("year_month", F.date_format("date_key", "yyyy-MM")))

# Fact table: one row per order line
fact_sales = (det
    .join(orders.select("OrderID", "CustomerID", "EmployeeID", "order_date"), "OrderID")
    .select(F.col("OrderID").alias("order_id"),
            F.col("CustomerID").alias("customer_id"),
            F.col("ProductID").alias("product_id"),
            F.col("EmployeeID").alias("employee_id"),
            F.col("order_date").alias("date_key"),
            F.col("Quantity").alias("quantity"),
            F.col("UnitPrice").alias("unit_price"),
            F.col("Discount").alias("discount"))
    .withColumn("line_total",
        F.round(F.col("quantity") * F.col("unit_price") * (1 - F.col("discount")), 2)))

for name, df in [("dim_customer", dim_customer), ("dim_product", dim_product),
                 ("dim_employee", dim_employee), ("dim_date", dim_date),
                 ("fact_sales", fact_sales)]:
    df.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable(name)

StatementMeta(, 57f7e8f0-4526-437a-b6ea-65e138918c73, 8, Finished, Available, Finished, False)

In [7]:
print(spark.table("fact_sales").count())                     # about 2155
print(spark.table("fact_sales").filter("date_key IS NULL").count())   # 0
spark.table("dim_product").filter("category IS NULL").count() 

StatementMeta(, 57f7e8f0-4526-437a-b6ea-65e138918c73, 9, Finished, Available, Finished, False)

2155
0


0